# Week 4 - Conditioning in Diffusion: TODO 4 Visual Check

This notebook is a visualization-first check for **TODO 4** in `notes/w04_conditioning_in_diffusion.md`.

Goal: corroborate the posterior-mean theory with clear plots and concise metrics (not coding drills).

We use the 1D Gaussian forward process:

$$
x_0 \sim \mathcal{N}(\mu_0, s_0^2), \qquad
\epsilon \sim \mathcal{N}(0,1), \qquad
x_t = \alpha_t x_0 + \sigma_t \epsilon.
$$

From the notes, the posterior mean is affine:

$$
m_t(x_t)=\mathbb{E}[x_0\mid x_t]=A_t x_t + b_t,
$$

with

$$
A_t=\frac{\alpha_t s_0^2}{\alpha_t^2 s_0^2 + \sigma_t^2},
\qquad
b_t=\frac{\mu_0\sigma_t^2}{\alpha_t^2 s_0^2 + \sigma_t^2}.
$$

## Outline

1. **TODO 4.1** - Simulate pairs $(x_0, x_t)$.
2. **TODO 4.2** - Fit $\hat{x}_0 = \hat{a}x_t + \hat{b}$ and compare with theory.
3. Compare empirical MSE against the theoretical variance floor.
4. Sweep noise level to visualize how conditioning changes.
5. Use the final prompt to write your TODO 4.3 note.

---

## 0) Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def posterior_coeffs(alpha_t: float, sigma_t: float, mu0: float, s0: float):
    # Return posterior mean coefficients A_t, b_t and posterior variance v_t.
    denom = alpha_t**2 * s0**2 + sigma_t**2
    A_t = alpha_t * s0**2 / denom
    b_t = mu0 * sigma_t**2 / denom
    v_t = sigma_t**2 * s0**2 / denom
    return A_t, b_t, v_t


# Reproducibility
seed = 7
rng = np.random.default_rng(seed)

# Forward-process parameters
mu0 = 1.0
s0 = 1.2
alpha_t = 0.8
sigma_t = 0.9
n = 200_000

A_t, b_t, v_t = posterior_coeffs(alpha_t, sigma_t, mu0, s0)

print(f"seed={seed}, n={n}")
print(f"mu0={mu0}, s0={s0}, alpha_t={alpha_t}, sigma_t={sigma_t}")
print(f"Theoretical posterior: m_t(x_t) = {A_t:.4f} * x_t + {b_t:.4f}")
print(f"Theoretical posterior variance v_t = {v_t:.4f}")

assert alpha_t > 0 and sigma_t > 0 and s0 > 0
assert v_t > 0

---

## 1) TODO 4.1 - Simulate pairs $(x_0, x_t)$

In [ ]:
x0 = rng.normal(loc=mu0, scale=s0, size=n)
eps = rng.normal(loc=0.0, scale=1.0, size=n)
xt = alpha_t * x0 + sigma_t * eps

# Quick moment check for x_t
mean_xt_emp = np.mean(xt)
var_xt_emp = np.var(xt)

mean_xt_theory = alpha_t * mu0
var_xt_theory = alpha_t**2 * s0**2 + sigma_t**2

print(f"E[x_t]    empirical={mean_xt_emp:.4f} | theory={mean_xt_theory:.4f}")
print(f"Var[x_t]  empirical={var_xt_emp:.4f} | theory={var_xt_theory:.4f}")

assert abs(mean_xt_emp - mean_xt_theory) < 0.02
assert abs(var_xt_emp - var_xt_theory) < 0.03

In [ ]:
idx = rng.choice(n, size=4000, replace=False)

plt.figure(figsize=(7, 5))
plt.scatter(xt[idx], x0[idx], s=8, alpha=0.18)
plt.xlabel(r"$x_t$")
plt.ylabel(r"$x_0$")
plt.title("Simulated pairs: noisy observation vs clean sample")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

---

## 2) TODO 4.2 - Fit $\hat{x}_0 = \hat{a}x_t + \hat{b}$ and compare with $m_t(x_t)$

In [ ]:
# OLS fit for x0 ~ a * xt + b
a_hat = np.cov(xt, x0, ddof=0)[0, 1] / np.var(xt)
b_hat = np.mean(x0) - a_hat * np.mean(xt)

print(f"a_hat={a_hat:.4f} vs A_t(theory)={A_t:.4f}")
print(f"b_hat={b_hat:.4f} vs b_t(theory)={b_t:.4f}")

assert abs(a_hat - A_t) < 0.015
assert abs(b_hat - b_t) < 0.02

In [ ]:
x_grid = np.linspace(np.percentile(xt, 1), np.percentile(xt, 99), 200)
mean_theory = A_t * x_grid + b_t
mean_fit = a_hat * x_grid + b_hat

# Binned empirical conditional means E[x0 | xt in bin]
bins = np.linspace(np.percentile(xt, 1), np.percentile(xt, 99), 26)
bin_id = np.digitize(xt, bins)
xb, yb = [], []
for k in range(1, len(bins)):
    mask = bin_id == k
    if np.sum(mask) > 200:
        xb.append(np.mean(xt[mask]))
        yb.append(np.mean(x0[mask]))

plt.figure(figsize=(7, 5))
plt.scatter(xb, yb, s=25, label="Empirical bin means")
plt.plot(x_grid, mean_theory, linewidth=2.5, label=r"Theory $m_t(x_t)=A_t x_t+b_t$")
plt.plot(x_grid, mean_fit, linestyle="--", linewidth=2, label=r"Fitted $\hat{a}x_t+\hat{b}$")
plt.xlabel(r"$x_t$")
plt.ylabel(r"Predicted/average $x_0$")
plt.title("Posterior mean check: theory vs fit vs empirical bins")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

---

## 3) Error Floor Check

Under MSE, the best predictor is $m_t(x_t)=\mathbb{E}[x_0\mid x_t]$, and the best achievable error is

$$
\mathbb{E}[\mathrm{Var}(x_0\mid x_t)] = v_t
$$

(in this Gaussian case, $v_t$ is constant in $x_t$).

In [ ]:
pred_theory = A_t * xt + b_t
pred_fit = a_hat * xt + b_hat
pred_naive = np.full_like(x0, fill_value=mu0)

mse_theory = np.mean((x0 - pred_theory) ** 2)
mse_fit = np.mean((x0 - pred_fit) ** 2)
mse_naive = np.mean((x0 - pred_naive) ** 2)

print(f"MSE(theory posterior mean) = {mse_theory:.4f}")
print(f"MSE(fitted linear model)   = {mse_fit:.4f}")
print(f"MSE(naive constant mu0)    = {mse_naive:.4f}")
print(f"Theoretical floor v_t      = {v_t:.4f}")

assert mse_theory <= mse_naive
assert abs(mse_theory - v_t) < 0.02

---

## 4) Noise Sweep for Intuition

We now vary $\sigma_t$ to see how posterior coefficients move:

- $A_t$ should decrease as noise grows (less trust in $x_t$).
- $b_t$ should move toward $\mu_0$ (more pull toward prior mean).
- $v_t$ should increase toward $s_0^2$ (more uncertainty).

In [ ]:
sigma_values = np.array([0.1, 0.3, 0.6, 1.0, 2.0, 4.0])
A_vals, b_vals, v_vals = [], [], []

for s in sigma_values:
    A_s, b_s, v_s = posterior_coeffs(alpha_t, s, mu0, s0)
    A_vals.append(A_s)
    b_vals.append(b_s)
    v_vals.append(v_s)
    print(f"sigma={s:>3.1f} | A_t={A_s:.4f}, b_t={b_s:.4f}, v_t={v_s:.4f}")

A_vals = np.array(A_vals)
b_vals = np.array(b_vals)
v_vals = np.array(v_vals)

assert np.all(A_vals[:-1] > A_vals[1:])
assert np.all(v_vals[:-1] < v_vals[1:])

fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
ax[0].plot(sigma_values, A_vals, marker="o")
ax[0].set_title(r"$A_t$ vs $\sigma_t$")
ax[0].set_xlabel(r"$\sigma_t$")
ax[0].grid(alpha=0.2)

ax[1].plot(sigma_values, b_vals, marker="o")
ax[1].axhline(mu0, linestyle="--", linewidth=1.5, color="tab:gray", label=r"$\mu_0$")
ax[1].set_title(r"$b_t$ vs $\sigma_t$")
ax[1].set_xlabel(r"$\sigma_t$")
ax[1].legend()
ax[1].grid(alpha=0.2)

ax[2].plot(sigma_values, v_vals, marker="o")
ax[2].axhline(s0**2, linestyle="--", linewidth=1.5, color="tab:gray", label=r"$s_0^2$")
ax[2].set_title(r"$v_t$ vs $\sigma_t$")
ax[2].set_xlabel(r"$\sigma_t$")
ax[2].legend()
ax[2].grid(alpha=0.2)

plt.tight_layout()
plt.show()

---

## Common Pitfall

If you only inspect one noisy sample $x_t$, it feels like denoising should be exact. But the objective is in expectation over the data distribution, and the decomposition shows an unavoidable variance floor.

## Optional Extension

Repeat the notebook with a different $\alpha_t$ (for example, $0.4$ vs $0.95$) and compare how quickly $A_t$ collapses as $\sigma_t$ grows.

---

## Quick Exercise (for TODO 4.3 write-up)

Write 2-3 lines in your notes addressing:

1. Does the fitted line $\hat{a}x_t+\hat{b}$ match the theoretical posterior mean $A_t x_t+b_t$?
2. Is the observed MSE near the theoretical floor $v_t$?
3. If there is any mismatch, is it from finite-sample noise or a theory issue?

Answer check (expected): agreement should be close, and small differences should be finite-sample effects.